# IPL Match Prediction

Goal: predict whether **team1** will win the match using match metadata.

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('matches.csv')
df.head(3)

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar


In [3]:
df

,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1090,1426307,2024,Hyderabad,2024-05-19,League,Abhishek Sharma,"Rajiv Gandhi International Stadium, Uppal, Hyd...",Punjab Kings,Sunrisers Hyderabad,Punjab Kings,bat,Sunrisers Hyderabad,wickets,4.0,215.0,20.0,N,NaN,Nitin Menon,VK Sharma
1091,1426309,2024,Ahmedabad,2024-05-21,Qualifier 1,MA Starc,"Narendra Modi Stadium, Ahmedabad",Sunrisers Hyderabad,Kolkata Knight Riders,Sunrisers Hyderabad,bat,Kolkata Knight Riders,wickets,8.0,160.0,20.0,N,NaN,AK Chaudhary,R Pandit
1092,1426310,2024,Ahmedabad,2024-05-22,Eliminator,R Ashwin,"Narendra Modi Stadium, Ahmedabad",Royal Challengers Bengaluru,Rajasthan Royals,Rajasthan Royals,field,Rajasthan Royals,wickets,4.0,173.0,20.0,N,NaN,KN Ananthapadmanabhan,MV Saidharshan Kumar
1093,1426311,2024,Chennai,2024-05-24,Qualifier 2,Shahbaz Ahmed,"MA Chidambaram Stadium, Chepauk, Chennai",Sunrisers Hyderabad,Rajasthan Royals,Rajasthan Royals,field,Sunrisers Hyderabad,runs,36.0,176.0,20.0,N,NaN,Nitin Menon,VK Sharma


In [ ]:
print('Columns:', df.columns.tolist())
print('Shape:', df.shape)
df.dtypes

In [ ]:
cols_to_drop = [
    'id', 'method', 'umpire1', 'umpire2', 'result_margin',
    'super_over', 'target_overs', 'player_of_match',
    'result', 'target_runs', 'date'
]
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
df.head(3)

In [ ]:
# Team name standardization
team_mapping = {
    'Mumbai Indians': 'MI',
    'Royal Challengers Bangalore': 'RCB',
    'Royal Challengers Bengaluru': 'RCB',
    'Chennai Super Kings': 'CSK',
    'Kolkata Knight Riders': 'KKR',
    'Delhi Daredevils': 'DC',
    'Delhi Capitals': 'DC',
    'Kings XI Punjab': 'PBKS',
    'Punjab Kings': 'PBKS',
    'Rajasthan Royals': 'RR',
    'Sunrisers Hyderabad': 'SRH',
    'Deccan Chargers': 'HDC',
    'Gujarat Titans': 'GT',
    'Lucknow Super Giants': 'LSG',
    'Rising Pune Supergiant': 'RPS',
    'Rising Pune Supergiants': 'RPS',
    'Kochi Tuskers Kerala': 'KTK',
    'Pune Warriors': 'PWI',
    'Gujarat Lions': 'GL'
}

team_cols = ['team1', 'team2', 'winner', 'toss_winner']
for col in team_cols:
    if col in df.columns:
        df[col] = df[col].map(team_mapping).fillna(df[col])

In [ ]:
# Drop rows without a winner
df.dropna(subset=['winner'], inplace=True)

# Binary target: did team1 win?
df['target'] = (df['winner'] == df['team1']).astype(int)

In [ ]:
# Clean city / venue mapping
venue_to_city_map = {
    'M Chinnaswamy Stadium': 'Bangalore',
    'M.Chinnaswamy Stadium': 'Bangalore',
    'M Chinnaswamy Stadium, Bengaluru': 'Bangalore',
    'Punjab Cricket Association Stadium, Mohali': 'Mohali',
    'Punjab Cricket Association IS Bindra Stadium, Mohali': 'Mohali',
    'Punjab Cricket Association IS Bindra Stadium': 'Mohali',
    'Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh': 'Mohali',
    'Feroz Shah Kotla': 'Delhi',
    'Arun Jaitley Stadium': 'Delhi',
    'Arun Jaitley Stadium, Delhi': 'Delhi',
    'Wankhede Stadium': 'Mumbai',
    'Wankhede Stadium, Mumbai': 'Mumbai',
    'Dr DY Patil Sports Academy': 'Mumbai',
    'Dr DY Patil Sports Academy, Mumbai': 'Mumbai',
    'Brabourne Stadium': 'Mumbai',
    'Brabourne Stadium, Mumbai': 'Mumbai',
    'Eden Gardens': 'Kolkata',
    'Eden Gardens, Kolkata': 'Kolkata',
    'Sawai Mansingh Stadium': 'Jaipur',
    'Sawai Mansingh Stadium, Jaipur': 'Jaipur',
    'Rajiv Gandhi International Stadium, Uppal': 'Hyderabad',
    'Rajiv Gandhi International Stadium': 'Hyderabad',
    'Rajiv Gandhi International Stadium, Uppal, Hyderabad': 'Hyderabad',
    'MA Chidambaram Stadium, Chepauk': 'Chennai',
    'MA Chidambaram Stadium': 'Chennai',
    'MA Chidambaram Stadium, Chepauk, Chennai': 'Chennai',
    'Newlands': 'Cape Town',
    "St George's Park": 'Gqeberha',
    'Kingsmead': 'Durban',
    'SuperSport Park': 'Centurion',
    'Buffalo Park': 'East London',
    'New Wanderers Stadium': 'Johannesburg',
    'De Beers Diamond Oval': 'Kimberley',
    'OUTsurance Oval': 'Bloemfontein',
    'Sardar Patel Stadium, Motera': 'Ahmedabad',
    'Narendra Modi Stadium, Ahmedabad': 'Ahmedabad',
    'Barabati Stadium': 'Cuttack',
    'Vidarbha Cricket Association Stadium, Jamtha': 'Nagpur',
    'Himachal Pradesh Cricket Association Stadium': 'Dharamsala',
    'Himachal Pradesh Cricket Association Stadium, Dharamsala': 'Dharamsala',
    'Nehru Stadium': 'Kochi',
    'Holkar Cricket Stadium': 'Indore',
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium': 'Visakhapatnam',
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam': 'Visakhapatnam',
    'Subrata Roy Sahara Stadium': 'Pune',
    'Maharashtra Cricket Association Stadium': 'Pune',
    'Maharashtra Cricket Association Stadium, Pune': 'Pune',
    'Shaheed Veer Narayan Singh International Stadium': 'Raipur',
    'JSCA International Stadium Complex': 'Ranchi',
    'Sheikh Zayed Stadium': 'Abu Dhabi',
    'Zayed Cricket Stadium, Abu Dhabi': 'Abu Dhabi',
    'Sharjah Cricket Stadium': 'Sharjah',
    'Dubai International Cricket Stadium': 'Dubai',
    'Saurashtra Cricket Association Stadium': 'Rajkot',
    'Green Park': 'Kanpur',
    'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow': 'Lucknow',
    'Barsapara Cricket Stadium, Guwahati': 'Guwahati',
    'Maharaja Yadavindra Singh International Cricket Stadium, Mullanpur': 'Mullanpur'
}

df['city'] = df['city'].map(venue_to_city_map).fillna(df['city'])
if 'venue' in df.columns:
    df['city'] = df['city'].fillna(df['venue'].map(venue_to_city_map))
df['city'] = df['city'].fillna('Unknown')

In [ ]:
# Season year extraction
df['season_year'] = df['season'].astype(str).str.extract(r'(\d{4})').astype(int)
df.drop(columns=['season'], inplace=True)

In [ ]:
match_type_map = {
    'League': 0,
    '3rd Place Play-Off': 1,
    'Elimination Final': 2,
    'Eliminator': 2,
    'Qualifier 1': 3,
    'Qualifier 2': 4,
    'Semi Final': 5,
    'Final': 6
}
df['match_type'] = df['match_type'].map(match_type_map).fillna(0).astype(int)

toss_map = {'field': 0, 'bat': 1}
df['toss_decision'] = df['toss_decision'].map(toss_map).fillna(0).astype(int)

df['is_toss_winner_team1'] = (df['toss_winner'] == df['team1']).astype(int)

# Bat-first flag: True when team1 bats first
df['t1_bat_first'] = (
    ((df['is_toss_winner_team1'] == 1) & (df['toss_decision'] == 1)) |
    ((df['is_toss_winner_team1'] == 0) & (df['toss_decision'] == 0))
).astype(int)

df.head(3)

## Feature Engineering: Win Rates & Head-to-Head

In [ ]:
# Sort by season_year so historical stats don't leak future info
df = df.sort_values('season_year').reset_index(drop=True)

# Overall team win rate (cumulative, excluding current match)
team_stats = {}
for team in set(df['team1']).union(set(df['team2'])):
    team_matches = df[(df['team1'] == team) | (df['team2'] == team)]
    team_wins = team_matches[team_matches['winner'] == team].shape[0]
    team_stats[team] = team_wins / max(team_matches.shape[0], 1)

df['team1_win_rate'] = df['team1'].map(team_stats).fillna(0.5)
df['team2_win_rate'] = df['team2'].map(team_stats).fillna(0.5)
df['win_rate_diff'] = df['team1_win_rate'] - df['team2_win_rate']

In [ ]:
# Head-to-head win rate for team1 vs team2
def compute_h2h_rates(dataframe):
    h2h = {}
    for _, row in dataframe.iterrows():
        t1, t2, w = row['team1'], row['team2'], row['winner']
        key = tuple(sorted([t1, t2]))
        if key not in h2h:
            h2h[key] = {'t1': t1, 't1_wins': 0, 'total': 0}
        h2h[key]['total'] += 1
        if w == t1:
            h2h[key]['t1_wins'] += 1
    
    rates = {}
    for key, val in h2h.items():
        rates[key] = val['t1_wins'] / max(val['total'], 1)
    return rates

# Compute on full data for mapping
h2h_rates = compute_h2h_rates(df)
df['h2h_team1_win_rate'] = df.apply(
    lambda r: h2h_rates.get(tuple(sorted([r['team1'], r['team2']])), 0.5), axis=1
)

In [ ]:
# City win rate for team1
city_stats = {}
for city in df['city'].unique():
    city_matches = df[df['city'] == city]
    if city_matches.shape[0] == 0:
        city_stats[city] = 0.5
        continue
    city_wins = city_matches[city_matches['winner'] == city_matches['team1']].shape[0]
    city_stats[city] = city_wins / city_matches.shape[0]

df['city_team1_win_rate'] = df['city'].map(city_stats).fillna(0.5)

In [ ]:
# Final feature selection
feature_cols = [
    'team1', 'team2', 'toss_winner', 'city',
    'match_type', 'toss_decision', 'is_toss_winner_team1',
    't1_bat_first', 'team1_win_rate', 'team2_win_rate',
    'win_rate_diff', 'h2h_team1_win_rate', 'city_team1_win_rate'
]

X = df[feature_cols].copy()
y = df['target'].copy()
X.head(3)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import TargetEncoder, OrdinalEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, precision_recall_fscore_support
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
)
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Split FIRST to prevent leakage
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Target encode high-cardinality categoricals using train-only stats
te_city = TargetEncoder(smooth='auto', cv=5)
te_t1 = TargetEncoder(smooth='auto', cv=5)
te_t2 = TargetEncoder(smooth='auto', cv=5)
te_tw = TargetEncoder(smooth='auto', cv=5)

x_train['city_enc'] = te_city.fit_transform(x_train[['city']], y_train)
x_test['city_enc'] = te_city.transform(x_test[['city']])

x_train['team1_enc'] = te_t1.fit_transform(x_train[['team1']], y_train)
x_test['team1_enc'] = te_t1.transform(x_test[['team1']])

x_train['team2_enc'] = te_t2.fit_transform(x_train[['team2']], y_train)
x_test['team2_enc'] = te_t2.transform(x_test[['team2']])

x_train['toss_winner_enc'] = te_tw.fit_transform(x_train[['toss_winner']], y_train)
x_test['toss_winner_enc'] = te_tw.transform(x_test[['toss_winner']])

# Drop raw categorical columns
drop_cols = ['city', 'team1', 'team2', 'toss_winner']
x_train = x_train.drop(columns=drop_cols)
x_test = x_test.drop(columns=drop_cols)

print('Train shape:', x_train.shape)
print('Test shape:', x_test.shape)
print('Features:', x_train.columns.tolist())

In [ ]:
# Scale for linear models
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

# Models
models = {
    'Logistic Regression': (
        LogisticRegression(C=0.1, max_iter=3000, solver='liblinear', random_state=42),
        True
    ),
    'Random Forest': (
        RandomForestClassifier(
            n_estimators=400, max_depth=10, min_samples_split=5,
            min_samples_leaf=2, random_state=42, n_jobs=-1
        ),
        False
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, random_state=42
        ),
        False
    ),
    'XGBoost': (
        XGBClassifier(
            n_estimators=400, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, eval_metric='logloss', n_jobs=-1
        ),
        False
    )
}

results = {}
for name, (model, needs_scaling) in models.items():
    X_tr = x_train_scaled if needs_scaling else x_train
    X_te = x_test_scaled if needs_scaling else x_test
    
    model.fit(X_tr, y_train)
    preds = model.predict(X_te)
    probs = model.predict_proba(X_te)[:, 1]
    
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    p, r, f1, _ = precision_recall_fscore_support(y_test, preds, average='binary', zero_division=0)
    
    results[name] = {
        'Accuracy': round(acc, 4),
        'ROC-AUC': round(auc, 4),
        'Precision': round(p, 4),
        'Recall': round(r, 4),
        'F1': round(f1, 4)
    }

results_df = pd.DataFrame(results).T
results_df

In [ ]:
# Cross-validation comparison (5-fold stratified)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
for name, (model, needs_scaling) in models.items():
    X_cv = x_train_scaled if needs_scaling else x_train
    scores = cross_val_score(model, X_cv, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_results[name] = {
        'CV Accuracy Mean': round(scores.mean(), 4),
        'CV Std': round(scores.std(), 4)
    }

cv_df = pd.DataFrame(cv_results).T
cv_df

In [ ]:
# Best single model on CV accuracy
best_model_name = cv_df['CV Accuracy Mean'].idxmax()
print('Best model by CV accuracy:', best_model_name)

best_model, needs_scaling = models[best_model_name]
X_tr = x_train_scaled if needs_scaling else x_train
X_te = x_test_scaled if needs_scaling else x_test
best_model.fit(X_tr, y_train)
y_pred = best_model.predict(X_te)
y_prob = best_model.predict_proba(X_te)[:, 1]

print('\n--- Classification Report ---')
print(classification_report(y_test, y_pred, target_names=['Team2 wins', 'Team1 wins']))
print('\n--- Confusion Matrix ---')
print(pd.DataFrame(confusion_matrix(y_test, y_pred),
       index=['Actual Team2', 'Actual Team1'],
       columns=['Pred Team2', 'Pred Team1']))

In [ ]:
# Ensemble Voting Classifier (soft voting)
# Train on scaled data for LR, original for tree models
# VotingClassifier expects same input; use scaled for all

lr_v = LogisticRegression(C=0.1, max_iter=3000, solver='liblinear', random_state=42)
rf_v = RandomForestClassifier(n_estimators=400, max_depth=10, min_samples_split=5,
                               min_samples_leaf=2, random_state=42, n_jobs=-1)
gb_v = GradientBoostingClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                                   subsample=0.8, random_state=42)

voting = VotingClassifier(
    estimators=[('lr', lr_v), ('rf', rf_v), ('gb', gb_v)],
    voting='soft', n_jobs=-1
)

voting.fit(x_train_scaled, y_train)
v_pred = voting.predict(x_test_scaled)
v_prob = voting.predict_proba(x_test_scaled)[:, 1]

v_acc = accuracy_score(y_test, v_pred)
v_auc = roc_auc_score(y_test, v_prob)
vp, vr, vf1, _ = precision_recall_fscore_support(y_test, v_pred, average='binary', zero_division=0)

print('--- Voting Ensemble ---')
print(f'Accuracy : {v_acc:.4f}')
print(f'ROC-AUC : {v_auc:.4f}')
print(f'Precision: {vp:.4f}')
print(f'Recall   : {vr:.4f}')
print(f'F1       : {vf1:.4f}')

In [ ]:
# Feature importance from Random Forest
rf_model = models['Random Forest'][0]
rf_model.fit(x_train, y_train)  # fit on unscaled for feature importance

fi = pd.DataFrame({
    'feature': x_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

fi

In [ ]:
# XGBoost feature importance
xgb_model = models['XGBoost'][0]
xgb_model.fit(x_train, y_train)

xgb_fi = pd.DataFrame({
    'feature': x_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

xgb_fi